# Unidad 4 — Funciones, Errores y Programación Funcional

**Programación Orientada a Objetos — ISTEA**

Las funciones permiten encapsular lógica reutilizable. El manejo de errores hace que los programas
sean robustos. Los conceptos de programación funcional (lambda, map, filter, reduce) compactan
transformaciones sobre colecciones.

**Temas que practico en esta unidad:**
- Manejar excepciones con try/except/else/finally y raise
- Definir funciones con *args y **kwargs
- Usar funciones lambda
- Crear generadores con yield
- Escribir decoradores
- Aplicar map, filter y reduce

*Consignas de la cátedra. Resoluciones desarrolladas durante la cursada.*

## Ejercicios

### 🟢 Ejercicio 1 — División segura
Función que divide dos números manejando ZeroDivisionError y TypeError.


In [1]:
# 🎯 EJERCICIO 1: División segura con manejo de errores
def dividir_seguro(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Error: no se puede dividir por cero")
    except TypeError:
        print(f"Error: no se puede dividir {type(a).__name__} por {type(b).__name__}")
    return None

# Pruebas:
print(dividir_seguro(10, 2))    # → 5.0
print(dividir_seguro(10, 0))    # → None (con mensaje de error)
print(dividir_seguro("a", 2))   # → None (con mensaje de error)

5.0
Error: no se puede dividir por cero
None
Error: no se puede dividir str por int
None


### 🟡 Ejercicio 2 — Función con *args para estadísticas
Función que calcula media, mínimo y máximo de cualquier cantidad de valores.


In [2]:
# 🎯 EJERCICIO 2: Estadísticas con *args
def estadisticas(*valores):
    if not valores:          # si no recibe nada, no hay nada que calcular
        return None
    return {
        "media": round(sum(valores) / len(valores), 2),
        "minimo": min(valores),
        "maximo": max(valores),
    }

# Pruebas:
print(estadisticas(4, 7, 2, 9, 1, 5))
# Resultado esperado: {'media': 4.67, 'minimo': 1, 'maximo': 9}

{'media': 4.67, 'minimo': 1, 'maximo': 9}


### 🔴 Ejercicio 3 — Decorador @cronometrar
Decorador que imprime el tiempo de ejecución de cualquier función.


In [3]:
# 🎯 EJERCICIO 3: Decorador cronometrar
import time

def cronometrar(funcion):
    def wrapper(*args, **kwargs):
        inicio = time.perf_counter()
        resultado = funcion(*args, **kwargs)
        print(f"Tiempo de ejecución: {(time.perf_counter() - inicio) * 1000:.2f} ms")
        return resultado
    return wrapper

@cronometrar
def esperar_un_poco():
    time.sleep(0.1)
    return "listo"

# Prueba
resultado = esperar_un_poco()
print(f"Retorno de la función: {resultado}")

Tiempo de ejecución: 100.09 ms
Retorno de la función: listo


### 🟡 Ejercicio 4 — Generador de Fibonacci
Generador que produce números de Fibonacci hasta n.


In [4]:
# 🎯 EJERCICIO 4: Generador de Fibonacci (Explicado paso a paso)
def fibonacci_hasta(n):
    a, b = 0, 1
    while a <= n:
        yield a           # entrega el valor y la función queda "pausada" acá
        a, b = b, a + b   # cuando le piden otro, sigue desde esta línea

print("Iniciando iteración sobre el generador:")
for numero in fibonacci_hasta(34):
    print(f"-> El generador me entregó: {numero}")
    # Después de este print, el 'for' vuelve arriba y le dice al generador: "¡Dame otro!"

Iniciando iteración sobre el generador:
-> El generador me entregó: 0
-> El generador me entregó: 1
-> El generador me entregó: 1
-> El generador me entregó: 2
-> El generador me entregó: 3
-> El generador me entregó: 5
-> El generador me entregó: 8
-> El generador me entregó: 13
-> El generador me entregó: 21
-> El generador me entregó: 34


### 🔴 Ejercicio 5 — map + filter + lambda
De una lista de strings, conservar solo los de longitud > 5 y convertirlos a mayúsculas.


In [5]:
# 🎯 EJERCICIO 5: map + filter + lambda
palabras = ["Python", "es", "un", "lenguaje", "de", "programación", "muy", "poderoso"]

# filter se queda con las de más de 5 letras y map las pasa a mayúsculas
resultado = list(map(str.upper, filter(lambda p: len(p) > 5, palabras)))
print(resultado)

['PYTHON', 'LENGUAJE', 'PROGRAMACIÓN', 'PODEROSO']


## Desafío extra 🔴

Implementar un decorador `@reintentar(veces=3, espera=0.1)` que, si una función lanza una excepción,
la vuelve a llamar hasta `veces` veces con una pausa de `espera` segundos entre intentos.
Probar con una función que falla aleatoriamente.


In [6]:
import random
import time

def reintentar(veces=3, espera=0.1):
    # NIVEL 1: Recibe los argumentos del decorador (veces, espera)
    def decorador(funcion):
        # NIVEL 2: Recibe la función que vamos a decorar (conexion_inestable)
        def envoltura(*args, **kwargs):
            # NIVEL 3: La lógica que se ejecuta en cada llamada
            intentos_fallidos = 0

            while intentos_fallidos < veces:
                try:
                    # Intentamos ejecutar la función original
                    return funcion(*args, **kwargs)

                except Exception as e:
                    # Si falla, sumamos un intento
                    intentos_fallidos += 1
                    print(f"⚠️ Intento {intentos_fallidos} falló: {e}")

                    # Si todavía nos quedan intentos, esperamos y repetimos el bucle
                    if intentos_fallidos < veces:
                        print(f"   Reintentando en {espera}s...")
                        time.sleep(espera)
                    else:
                        # Si ya no hay más oportunidades, lanzamos la excepción
                        print("❌ Se agotaron los intentos.")
                        raise e

        return envoltura
    return decorador

# Aplicamos el decorador con parámetros específicos
@reintentar(veces=5, espera=0.05)
def conexion_inestable():
    '''Simula una conexión que falla el 70% de las veces.'''
    if random.random() < 0.7:
        raise ConnectionError("Fallo de conexión (simulado)")
    return "✅ Conexión exitosa"

# Prueba del sistema
random.seed(42) # Fijamos la semilla para que el resultado sea reproducible
try:
    resultado = conexion_inestable()
    print(resultado)
except ConnectionError:
    print("El sistema falló definitivamente tras todos los reintentos.")

⚠️ Intento 1 falló: Fallo de conexión (simulado)
   Reintentando en 0.05s...
⚠️ Intento 2 falló: Fallo de conexión (simulado)
   Reintentando en 0.05s...


⚠️ Intento 3 falló: Fallo de conexión (simulado)
   Reintentando en 0.05s...
⚠️ Intento 4 falló: Fallo de conexión (simulado)
   Reintentando en 0.05s...


✅ Conexión exitosa


## Resumen de conceptos

| Concepto | Sintaxis | Descripción |
|---|---|---|
| try/except | `try: ... except Exc as e:` | Captura excepciones |
| else | `else:` (tras except) | Solo si no hubo excepción |
| finally | `finally:` | Siempre se ejecuta |
| raise | `raise ValueError("msg")` | Lanza excepción |
| *args | `def f(*args):` | Tupla de args variables |
| **kwargs | `def f(**kwargs):` | Dict de kwargs variables |
| lambda | `lambda x: x*2` | Función anónima |
| yield | `def gen(): yield x` | Generador |
| Decorador | `@mi_deco` / `def deco(f):` | Modifica función |
| map | `map(func, iterable)` | Aplica función a cada elem |
| filter | `filter(func, iterable)` | Filtra por condición |
| reduce | `reduce(func, iterable)` | Acumula resultados |
